# 05 — Fetch Subscriptions

Pulls subscriptions from MySQL per `SUBSCRIPTION_QUERY` (currently scoped to
a single `AccountCode` for testing — see the query in step 2), then for each
one:

1. Decides which target account it belongs to (`resolve_target_accounts`,
   based on `Reference` — see step 3 below for the full priority order).
2. Looks up its real address + radius username from Voyager
   (`get_voyager_address`) — `SupplierServiceID` -> `GET .../fibre/v1/circuits/{id}`
   (gives `radiusUsers[0]` + `locationId`) -> `GET .../address-search/v3/addresses/id/{locationId}`
   (gives the actual street address, city, postcode, region). Region name
   is mapped to a 3-char ISO code via `NZ_Regions.xlsx`.

This is a data-prep step that calls out to Voyager (not OneBill) — no
OneBill API calls happen here. Output feeds both `06_Create_Addresses.ipynb`
and `07_Create_Subscription_Orders.ipynb` (the latter now uses the resolved
`radius_user` instead of `SubscriptionLabel`).

> **TODO**: confirm the `Reference` column name
> (`SUBSCRIPTION_REFERENCE_COLUMN` in `onebill_common.py`, currently
> `"Reference"`).


## 1. Setup

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from sqlalchemy import create_engine
from concurrent.futures import ThreadPoolExecutor

logger = get_logger("fetch_subscriptions")

# Use this to limit rows while testing. Set to None once ready for a full run.
TEST_ROW_LIMIT = None
BATCH_NUMBER =  os.environ["BATCH_NUMBER"]

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 27
python-dotenv could not parse statement starting at line 32
python-dotenv could not parse statement starting at line 38
python-dotenv could not parse statement starting at line 44


## 2. Pull every subscription from MySQL

In [2]:
assert BI_DATASTORE_URL, "DB_USERNAME/DB_PASSWORD/DB_HOST not set in .env"
engine = create_engine(BI_DATASTORE_URL)

SUBSCRIPTION_QUERY = '''
SELECT * FROM bi_datastore.billing_subscription
WHERE _DataSource = 'vBill'
AND AccountCode = '99965692'
AND SubscriptionEndDate IS NULL
AND SubscriptionUSN = 'V113105167'
ORDER BY RAND();
'''.strip()

df_subscriptions = pd.read_sql(SUBSCRIPTION_QUERY, con=engine)
logger.info(f"Loaded {len(df_subscriptions):,} subscriptions from MySQL")

if TEST_ROW_LIMIT is not None:
    df_subscriptions = df_subscriptions.head(TEST_ROW_LIMIT)  # Testing limiter — remove/raise for a full run.
    logger.info(f"TEST_ROW_LIMIT active — trimmed to {len(df_subscriptions):,} rows")

df_subscriptions.head()


2026-08-04 06:38:43,192 [INFO] Loaded 1 subscriptions from MySQL


,_rowid,_rowmodified,_sourceid,_DataSource,AccountCode,ServiceType,SubscriptionUSN,SubscriptionLabel,SubscriptionStartDate,SubscriptionEndDate,...,CircuitType,Server,CustomerSuppliedReference,_notforreports_VoyagerOrderHistory,_notforreports_LegacyServiceDescription,NextPlanCode,NextPlanStartDate,NextQuantity,NextCustomPrice,SalesAgentCode
0,4405092285,2026-08-02 01:00:49,266346,vBill,99965692,Broadband - Fibre,V113105167,24.51tekanawacr@williamsinternet.com,2026-07-16,None,...,UFB 100/20/2.5/2.5,None,WC - WC CHCH T9,CPP 3506895,None,None,None,None,None,None


## 3. Resolve target account for every subscription

`TargetAccountNumber` is resolved by `resolve_target_accounts()` (in
`onebill_common.py`, shared with `05_Fetch_Inactive_Subscriptions.ipynb` for
when that comes back into scope), in this priority order:

1. **Managed by Williams** — `Reference` contains the full phrase or the
   `MBW` abbreviation -> the shared `managed_by_williams` bucket account.
   Overrides own-account routing below, even if the subscription's own
   account exists.
2. **Fixed-reference accounts** — `Reference` matches one of
   `FIXED_REFERENCE_ACCOUNTS` (currently: Williams Real Estate, Toa Koura
   Limited, Design by Williams) -> that marker's own fixed, already-existing
   OneBill account. Same priority as step 1 — also overrides own-account
   routing.
3. **Own account** — matched by `AccountCode` against
   `04_Create_Accounts.ipynb`'s real results (`load_account_code_batch_map`).
   This is the actual OneBill `accountNumber` for that subscription's real
   account.
4. **Williams Corporation fallback** — if the subscription's own account has
   no successful (`created`/`exists`) row in `account_results` (not
   migrated, or failed), and it isn't caught by steps 1–2 either.

Run `04_Create_Accounts.ipynb` before this notebook if you haven't already.


In [3]:
own_account_map = load_account_code_batch_map()  # {AccountCode: AccountCode_Batch}, every account in 04's results
real_account_numbers = load_real_target_account_numbers()  # {"managed_by_williams": "...", "williams_corporation": "..."}

missing_keys = set(TARGET_ACCOUNTS) - set(real_account_numbers)
if missing_keys:
    logger.warning(
        f"No successful account_results row found for: {missing_keys} — "
        f"falling back to the TARGET_ACCOUNTS placeholder for those. "
        f"Run 04_Create_Accounts.ipynb (or check it for failures) before trusting this run."
    )

logger.info(f"{len(own_account_map):,} accounts available from 04_Create_Accounts.ipynb (own-account routing)")

reference_col = SUBSCRIPTION_REFERENCE_COLUMN if SUBSCRIPTION_REFERENCE_COLUMN in df_subscriptions.columns else None
if reference_col is None:
    logger.warning(
        f"Column '{SUBSCRIPTION_REFERENCE_COLUMN}' not found in df_subscriptions — "
        f"Managed-by-Williams / fixed-reference routing and the bucket fallback can't work without it. "
        f"Available columns: {list(df_subscriptions.columns)}"
    )
    df_subscriptions["CustomerSuppliedReference"] = None
else:
    df_subscriptions["CustomerSuppliedReference"] = df_subscriptions[reference_col]

df_subscriptions = resolve_target_accounts(df_subscriptions, own_account_map, real_account_numbers)

missing_own_account = (df_subscriptions["TargetAccountKey"] == "williams_corporation")
if missing_own_account.any():
    logger.warning(
        f"{missing_own_account.sum():,} subscriptions have no matching created/existing account in "
        f"04_Create_Accounts.ipynb's results and weren't caught by Managed-by-Williams or a "
        f"fixed-reference account — routed to the Williams Corporation bucket account instead."
    )

logger.info(df_subscriptions["TargetAccountKey"].value_counts(dropna=False).to_string())
df_subscriptions[["SubscriptionUSN", "AccountCode", "CustomerSuppliedReference", "TargetAccountKey", "TargetAccountNumber"]].head(20)


2026-08-04 06:38:43,339 [WARNING] No successful account_results row found for: {'williams_corporation'} — falling back to the TARGET_ACCOUNTS placeholder for those. Run 04_Create_Accounts.ipynb (or check it for failures) before trusting this run.
2026-08-04 06:38:43,342 [INFO] 2 accounts available from 04_Create_Accounts.ipynb (own-account routing)
2026-08-04 06:38:43,386 [WARNING] 1 subscriptions have no matching created/existing account in 04_Create_Accounts.ipynb's results and weren't caught by Managed-by-Williams or a fixed-reference account — routed to the Williams Corporation bucket account instead.
2026-08-04 06:38:43,392 [INFO] TargetAccountKey
williams_corporation    1


,SubscriptionUSN,AccountCode,CustomerSuppliedReference,TargetAccountKey,TargetAccountNumber
0,V113105167,99965692,WC - WC CHCH T9,williams_corporation,SR1602


## 4. Look up each subscription's real address + radius username (Voyager)

Two API calls per subscription, keyed on `SupplierServiceID`:
`fetch_voyager_circuit` -> `fetch_voyager_address`, wrapped by
`get_voyager_address`. Run in parallel (it's now real network calls, not
pure string parsing) via the same `ThreadPoolExecutor` pattern used in
`06_Create_Addresses.ipynb`.

Only ACTIVE subscriptions reach this point — inactive ones were split off
in step 2b and don't go through Voyager at all.

Subscriptions with no `SupplierServiceID`, or where either Voyager call
fails, come back with `parsed_ok = False` — flagged the same way unparsed
labels used to be, and skipped by `06_Create_Addresses.ipynb`.


In [4]:
if VOYAGER_CCP_KEY is None or VOYAGER_PARTNER_ID is None:
    logger.warning("VOYAGER_CCP_KEY / VOYAGER_PARTNER_ID not set — every Voyager lookup below will fail. Set them in .env.")

blank_supplier_ids = df_subscriptions["SupplierServiceID"].isna() | (df_subscriptions["SupplierServiceID"].astype(str).str.strip() == "")
if blank_supplier_ids.all():
    logger.warning(
        "SupplierServiceID is blank for EVERY subscription in this batch — the Voyager circuits lookup "
        "can't run at all without it, so every ParsedAddress_* field below will be None. Check the MySQL "
        "source data / SUBSCRIPTION_QUERY in step 2 before re-running."
    )
elif blank_supplier_ids.any():
    logger.warning(f"{blank_supplier_ids.sum():,} / {len(df_subscriptions):,} subscriptions have a blank SupplierServiceID")

voyager_session = new_voyager_session(max_workers=MAX_WORKERS)


def _lookup_row(supplier_service_id):
    return get_voyager_address(voyager_session, supplier_service_id)


logger.info(f"Looking up Voyager address for {len(df_subscriptions):,} subscriptions with {MAX_WORKERS} workers...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    voyager_results = list(executor.map(_lookup_row, df_subscriptions["SupplierServiceID"]))

existing_parsed_cols = [c for c in df_subscriptions.columns if c.startswith("ParsedAddress_")]
if existing_parsed_cols:
    df_subscriptions = df_subscriptions.drop(columns=existing_parsed_cols)  # safe to re-run this cell

address_parts = pd.DataFrame(voyager_results, index=df_subscriptions.index)
address_parts = address_parts.add_prefix("ParsedAddress_")
df_subscriptions = pd.concat([df_subscriptions, address_parts], axis=1)

unparsed = df_subscriptions[~df_subscriptions["ParsedAddress_parsed_ok"]]
if not unparsed.empty:
    logger.warning(f"{len(unparsed):,} subscriptions could not be resolved to a Voyager address")
    error_summary = (
        unparsed["ParsedAddress_error"].value_counts(dropna=False)
        .rename_axis("error").reset_index(name="count")
    )
    logger.info("Error breakdown:\n" + error_summary.to_string(index=False))

df_subscriptions[[
    "SubscriptionLabel", "SupplierServiceID",
    "ParsedAddress_addLine1", "ParsedAddress_addLine2", "ParsedAddress_city",
    "ParsedAddress_postcode", "ParsedAddress_region_iso", "ParsedAddress_region_code_raw", "ParsedAddress_radius_user",
    "ParsedAddress_parsed_ok", "ParsedAddress_error",
]].head(20)


2026-08-04 06:38:43,491 [INFO] Looking up Voyager address for 1 subscriptions with 3 workers...


,SubscriptionLabel,SupplierServiceID,ParsedAddress_addLine1,ParsedAddress_addLine2,ParsedAddress_city,ParsedAddress_postcode,ParsedAddress_region_iso,ParsedAddress_region_code_raw,ParsedAddress_radius_user,ParsedAddress_parsed_ok,ParsedAddress_error
0,24.51tekanawacr@williamsinternet.com,1643410431,24/51 TE KANAWA CRESCENT,HENDERSON,WAITAKERE,0610,AUK,AUK,24.51tekanawacr@williamsinternet.com,True,None


In [5]:
# Second pass: retry subscriptions that failed with a transient 5xx on the
# first pass, after a longer cooldown. 404s / "no SupplierServiceID" are
# permanent and intentionally skipped -- see voyager_second_pass() docstring.
SECOND_PASS_DELAY_SECONDS = 90  # bump this up if the same circuits still fail after 90s

df_subscriptions = voyager_second_pass(
    df_subscriptions,
    voyager_session,
    delay_seconds=SECOND_PASS_DELAY_SECONDS,
    max_workers=MAX_WORKERS,
)

unparsed = df_subscriptions[~df_subscriptions["ParsedAddress_parsed_ok"]]
if not unparsed.empty:
    error_summary = (
        unparsed["ParsedAddress_error"].value_counts(dropna=False)
        .rename_axis("error").reset_index(name="count")
    )
    logger.info(f"{len(unparsed):,} subscriptions still unresolved overall:\n" + error_summary.to_string(index=False))
else:
    logger.info("All subscriptions resolved.")

unparsed[["SubscriptionUSN", "SupplierServiceID", "ParsedAddress_error"]]


2026-08-04 06:38:48,102 [INFO] Voyager second pass: nothing to retry (no transient-5xx failures from the first pass).
2026-08-04 06:38:48,107 [INFO] All subscriptions resolved.


,SubscriptionUSN,SupplierServiceID,ParsedAddress_error


In [6]:
df_subscriptions["SupplierServiceID"] = df_subscriptions["SupplierServiceID"] + BATCH_NUMBER
df_subscriptions["ParsedAddress_radius_user"] = df_subscriptions["ParsedAddress_radius_user"] + BATCH_NUMBER
df_subscriptions["SubscriptionUSN"] = df_subscriptions["SubscriptionUSN"] + BATCH_NUMBER
df_subscriptions.head()

,_rowid,_rowmodified,_sourceid,_DataSource,AccountCode,ServiceType,SubscriptionUSN,SubscriptionLabel,SubscriptionStartDate,SubscriptionEndDate,...,ParsedAddress_addLine2,ParsedAddress_city,ParsedAddress_postcode,ParsedAddress_region_name,ParsedAddress_region_iso,ParsedAddress_region_code_raw,ParsedAddress_radius_user,ParsedAddress_location_id,ParsedAddress_parsed_ok,ParsedAddress_error
0,4405092285,2026-08-02 01:00:49,266346,vBill,99965692,Broadband - Fibre,V113105167,24.51tekanawacr@williamsinternet.com,2026-07-16,None,...,HENDERSON,WAITAKERE,0610,AUCKLAND REGION,AUK,AUK,24.51tekanawacr@williamsinternet.com,4b7a99d54c181b159bea5884091cc3b9a63fdef9,True,None


## 5. Save

In [7]:
save_df("subscriptions_resolved", df_subscriptions)


Saved 1 rows -> migration_data\05_subscriptions_resolved.csv
